# LeetCode 2144 — Minimum Cost of Buying Candies With Discount bằng Python

**Link đề:** https://leetcode.com/problems/minimum-cost-of-buying-candies-with-discount/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy, Sorting

```python
class Solution:
    def minimumCost(self, cost: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu luật khuyến mãi "mua 2 tặng 1" với điều kiện **kẹo tặng ≤ kẹo rẻ nhất trong 2 kẹo mua**.
2. Làm quen tư duy **tham lam (greedy)**: sắp xếp giảm dần, cứ 3 cái thì cái thứ 3 miễn phí.
3. Nắm `sorted(..., reverse=True)`, `.sort()`, slicing có bước `a[2::3]`, `enumerate`, `sum`.
4. Có **brute force** làm "trọng tài" (oracle) để kiểm tra greedy trên input nhỏ.
5. Chứng minh greedy đúng bằng **lập luận trao đổi (exchange argument)** ở mức trực giác.

## 1. Đề bài

Cửa hàng bán kẹo: **cứ mua 2 viên thì được tặng 1 viên thứ ba**. Viên được tặng phải có giá **≤ giá nhỏ nhất** của 2 viên đã mua.

`cost[i]` là giá viên kẹo `i`. Trả về **tổng tiền nhỏ nhất** để có **tất cả** các viên kẹo.

| `cost` | Kết quả | Cách mua tối ưu |
|---|:-:|---|
| `[1,2,3]` | `5` | mua 3 và 2, tặng 1 |
| `[6,5,7,9,2,2]` | `23` | mua 9,7 tặng 6; mua 5,2 tặng 2 → 9+7+5+2 |
| `[5,5]` | `10` | chỉ 2 viên, không có viên thứ ba để tặng |

**Constraints:** `1 <= len(cost) <= 100`, `1 <= cost[i] <= 100`.

Ví dụ đề lưu ý: với `[1,2,3]` **không** được mua 1 và 3 rồi lấy 2 miễn phí, vì `2 > min(1, 3) = 1`.

In [1]:
cost = [6, 5, 7, 9, 2, 2]
print("Tổng nếu không khuyến mãi:", sum(cost))
print("Đáp án đề:", 23, "-> tiết kiệm", sum(cost) - 23, "= 6 + 2 (hai viên được tặng)")

Tổng nếu không khuyến mãi: 31
Đáp án đề: 23 -> tiết kiệm 8 = 6 + 2 (hai viên được tặng)


## 2. Nền tảng Python cần biết

### 2.1. `sorted` và `.sort()`

- `sorted(a)` trả về **list mới** đã sắp tăng dần, `a` giữ nguyên.
- `sorted(a, reverse=True)` sắp **giảm dần**.
- `a.sort()` sắp **tại chỗ** (in-place), trả về `None`.

In [2]:
cost = [6, 5, 7, 9, 2, 2]
print("sorted tăng :", sorted(cost))
print("sorted giảm :", sorted(cost, reverse=True))
print("cost gốc    :", cost)
a = cost[:]          # bản sao
r = a.sort(reverse=True)
print("a sau sort  :", a, "| giá trị trả về của .sort():", r)

sorted tăng : [2, 2, 5, 6, 7, 9]
sorted giảm : [9, 7, 6, 5, 2, 2]
cost gốc    : [6, 5, 7, 9, 2, 2]
a sau sort  : [9, 7, 6, 5, 2, 2] | giá trị trả về của .sort(): None


### 2.2. Slicing có bước: `a[start::step]`

- `a[2::3]` lấy các phần tử chỉ số `2, 5, 8, ...` → đúng "mỗi viên thứ 3".
- `a[::3]` lấy chỉ số `0, 3, 6, ...`.

In [3]:
a = [9, 7, 6, 5, 2, 2, 1]
print("chỉ số :", list(range(len(a))))
print("a[2::3]:", a[2::3], "<- chỉ số 2, 5")
print("a[::3] :", a[::3])

chỉ số : [0, 1, 2, 3, 4, 5, 6]
a[2::3]: [6, 2] <- chỉ số 2, 5
a[::3] : [9, 5, 1]


### 2.3. `enumerate` và phép `%`

`for i, x in enumerate(a)` cho cả chỉ số `i` và giá trị `x`. Điều kiện `i % 3 == 2` đúng với `i = 2, 5, 8, ...` — cùng ý nghĩa với `a[2::3]`.

In [4]:
a = [9, 7, 6, 5, 2, 2, 1]
for i, x in enumerate(a):
    print(i, x, "i % 3 =", i % 3, "-> miễn phí" if i % 3 == 2 else "-> trả tiền")

0 9 i % 3 = 0 -> trả tiền
1 7 i % 3 = 1 -> trả tiền
2 6 i % 3 = 2 -> miễn phí
3 5 i % 3 = 0 -> trả tiền
4 2 i % 3 = 1 -> trả tiền
5 2 i % 3 = 2 -> miễn phí
6 1 i % 3 = 0 -> trả tiền


### 2.4. `sum` trên list / generator

`sum(a)` cộng mọi phần tử. `sum(x for i, x in enumerate(a) if i % 3 != 2)` cộng có điều kiện mà không cần tạo list trung gian (generator expression).

In [5]:
a = [9, 7, 6, 5, 2, 2]
print(sum(a))
print(sum(x for i, x in enumerate(a) if i % 3 != 2))
print(sum(a) - sum(a[2::3]))

31
23
23


### 2.5. Hiểu bài bằng tay

Muốn trả ít tiền nhất ⇔ muốn **tổng giá các viên được tặng lớn nhất**.

Thử với `[1,2,3]`:
- Mua 3 và 2 (min = 2) → được tặng viên ≤ 2 → lấy 1. Trả 5. ✔
- Mua 3 và 1 (min = 1) → chỉ được tặng viên ≤ 1 → không còn viên nào → trả 6.
- Mua 2 và 1 (min = 1) → viên 3 > 1, không tặng được → trả 6.

Quan sát: viên đắt nhất (3) **không bao giờ** được tặng (không có 2 viên nào ≥ nó). Viên đắt nhì cũng không. Viên đắt thứ ba thì **có thể** được tặng nếu ta mua 2 viên đắt nhất.

In [6]:
cost = [1, 2, 3]
for buy in [(3, 2), (3, 1), (2, 1)]:
    rest = [c for c in cost if c not in buy]
    free = [c for c in rest if c <= min(buy)]
    pay = sum(buy) + sum(rest) - (max(free) if free else 0)
    print(f"mua {buy}, min={min(buy)}, có thể tặng: {free} -> tổng trả {pay}")

mua (3, 2), min=2, có thể tặng: [1] -> tổng trả 5
mua (3, 1), min=1, có thể tặng: [] -> tổng trả 6
mua (2, 1), min=1, có thể tặng: [] -> tổng trả 6


## 3. Cách 1 — Brute force: thử mọi thứ tự

Ý tưởng "thử hết": xếp các viên kẹo theo **mọi hoán vị**, rồi gom lần lượt 3 viên một nhóm `(a, b, c)`:
- trả `a + b`, và trả thêm `c` nếu `c > min(a, b)` (không được tặng);
- nhóm cuối thiếu (1–2 viên) thì trả hết.

Mọi cách mua hợp lệ đều tương ứng với một hoán vị như vậy, nên lấy **min** trên mọi hoán vị là đáp án đúng. Dùng `itertools.permutations`.

In [7]:
from itertools import permutations
print(list(permutations([1, 2, 3])))

[(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]


In [8]:
from itertools import permutations

def cost_of_order(order):
    total = 0
    for g in range(0, len(order), 3):
        group = order[g:g + 3]
        if len(group) == 3:
            a, b, c = group
            total += a + b + (0 if c <= min(a, b) else c)
        else:
            total += sum(group)
    return total

def minimum_cost_bruteforce(cost: list[int]) -> int:
    return min(cost_of_order(p) for p in permutations(cost))

print(minimum_cost_bruteforce([1, 2, 3]))          # 5
print(minimum_cost_bruteforce([6, 5, 7, 9, 2, 2])) # 23
print(minimum_cost_bruteforce([5, 5]))             # 10

5
23
10


### Chi phí brute force

Có `n!` hoán vị, mỗi hoán vị tính `O(n)` → `O(n · n!)`. Với `n = 10` đã là 36 triệu bước; `n = 100` thì không tưởng. Nhưng brute force rất quý để **kiểm chứng** lời giải nhanh trên input nhỏ.

In [9]:
import math
for n in (3, 6, 8, 10, 12):
    print(f"n={n:>2}: n! = {math.factorial(n):,}")

n= 3: n! = 6
n= 6: n! = 720
n= 8: n! = 40,320
n=10: n! = 3,628,800
n=12: n! = 479,001,600


## 4. Nút thắt và insight (greedy)

**Nút thắt:** thử mọi cách gom nhóm quá nhiều. Cần một **quy tắc** chọn nhóm.

**Insight:** sắp giá **giảm dần** `c1 ≥ c2 ≥ c3 ≥ ...`.
- `c1`, `c2` chắc chắn phải trả (không có hai viên nào ≥ chúng để "che").
- Mua `c1`, `c2` → được tặng viên có giá ≤ `c2`; viên **đắt nhất** trong số đó là `c3`. Lấy `c3` miễn phí là lợi nhất.
- Lặp lại với phần còn lại `c4, c5, c6, ...`.

→ Sau khi sắp giảm dần, **mọi viên ở vị trí 2, 5, 8, ... (đếm từ 0) được miễn phí**.

Quan sát thêm: điều kiện "tặng ≤ min(hai viên mua)" luôn thỏa trong nhóm `(c[3t], c[3t+1], c[3t+2])` vì dãy giảm dần.

In [10]:
cost = [6, 5, 7, 9, 2, 2]
desc = sorted(cost, reverse=True)
print("giảm dần:", desc)
for g in range(0, len(desc), 3):
    print("nhóm:", desc[g:g + 3])

giảm dần: [9, 7, 6, 5, 2, 2]
nhóm: [9, 7, 6]
nhóm: [5, 2, 2]


## 5. Cách 2 — Lời giải chính: sắp giảm dần, bỏ qua mỗi viên thứ 3

Từng đoạn:

**Đoạn 1:** `cost = sorted(cost, reverse=True)`.

**Đoạn 2:** duyệt `i, x` bằng `enumerate`.

**Đoạn 3:** nếu `i % 3 == 2` → viên miễn phí, bỏ qua; ngược lại cộng `x` vào `total`.

**Đoạn 4:** trả `total`.

In [11]:
def minimum_cost_greedy(cost: list[int]) -> int:
    desc = sorted(cost, reverse=True)   # Đoạn 1
    total = 0
    for i, x in enumerate(desc):        # Đoạn 2
        if i % 3 == 2:                  # Đoạn 3: viên thứ 3 của nhóm -> free
            continue
        total += x
    return total                        # Đoạn 4

print(minimum_cost_greedy([1, 2, 3]))
print(minimum_cost_greedy([6, 5, 7, 9, 2, 2]))
print(minimum_cost_greedy([5, 5]))
print(minimum_cost_greedy([7]))

5
23
10
7


`continue` bỏ phần còn lại của lượt lặp hiện tại và nhảy sang lượt tiếp theo.

## 6. TRACE từng bước

`cost = [6,5,7,9,2,2]` → sắp giảm dần `[9,7,6,5,2,2]`.

| `i` | `x` | `i % 3` | Quyết định | `total` sau |
|:-:|:-:|:-:|---|:-:|
| 0 | 9 | 0 | trả | 9 |
| 1 | 7 | 1 | trả | 16 |
| 2 | 6 | 2 | **miễn phí** (6 ≤ min(9,7)) | 16 |
| 3 | 5 | 0 | trả | 21 |
| 4 | 2 | 1 | trả | 23 |
| 5 | 2 | 2 | **miễn phí** (2 ≤ min(5,2)) | 23 |

In [12]:
def trace_minimum_cost(cost):
    desc = sorted(cost, reverse=True)
    print("cost =", cost, "-> giảm dần", desc)
    total = 0
    for i, x in enumerate(desc):
        if i % 3 == 2:
            a, b = desc[i - 2], desc[i - 1]
            print(f"i={i}: x={x} miễn phí vì {x} <= min({a}, {b}) = {min(a, b)} | total={total}")
            continue
        total += x
        print(f"i={i}: x={x} trả tiền                       | total={total}")
    print("Kết quả:", total)
    return total

trace_minimum_cost([6, 5, 7, 9, 2, 2])

cost = [6, 5, 7, 9, 2, 2] -> giảm dần [9, 7, 6, 5, 2, 2]
i=0: x=9 trả tiền                       | total=9
i=1: x=7 trả tiền                       | total=16
i=2: x=6 miễn phí vì 6 <= min(9, 7) = 7 | total=16
i=3: x=5 trả tiền                       | total=21
i=4: x=2 trả tiền                       | total=23
i=5: x=2 miễn phí vì 2 <= min(5, 2) = 2 | total=23
Kết quả: 23


23

In [13]:
trace_minimum_cost([3, 3, 3, 3, 1])   # có giá trùng, và nhóm cuối thiếu

cost = [3, 3, 3, 3, 1] -> giảm dần [3, 3, 3, 3, 1]
i=0: x=3 trả tiền                       | total=3
i=1: x=3 trả tiền                       | total=6
i=2: x=3 miễn phí vì 3 <= min(3, 3) = 3 | total=6
i=3: x=3 trả tiền                       | total=9
i=4: x=1 trả tiền                       | total=10
Kết quả: 10


10

## 7. Code nộp LeetCode — Cách 2

In [14]:
class Solution:
    def minimumCost(self, cost: list[int]) -> int:
        cost = sorted(cost, reverse=True)
        return sum(x for i, x in enumerate(cost) if i % 3 != 2)

sol = Solution()
print(sol.minimumCost([1, 2, 3]), sol.minimumCost([6, 5, 7, 9, 2, 2]), sol.minimumCost([5, 5]))

5 23 10


## 8. Cách 3 — Tổng trừ phần được tặng (slicing có bước)

Tổng phải trả = `sum(cost) - (tổng các viên miễn phí)`, mà các viên miễn phí là `desc[2::3]`.

In [15]:
def minimum_cost_slice(cost: list[int]) -> int:
    desc = sorted(cost, reverse=True)
    return sum(desc) - sum(desc[2::3])

print(minimum_cost_slice([6, 5, 7, 9, 2, 2]))

23


In [16]:
class Solution:
    def minimumCost(self, cost: list[int]) -> int:
        cost.sort(reverse=True)
        return sum(cost) - sum(cost[2::3])

print(Solution().minimumCost([6, 5, 7, 9, 2, 2]))

23


Chú ý: bản này dùng `cost.sort(...)` nên **sửa luôn list đầu vào**. LeetCode chấp nhận, nhưng trong code thật nên dùng `sorted` nếu người gọi còn cần list gốc.

### Biến thể của cách 3: sắp **tăng dần** và duyệt từ cuối
Nếu sắp tăng dần, viên miễn phí là phần tử thứ 3, 6, 9... **tính từ cuối**: `asc[-3::-3]`.

In [17]:
asc = sorted([6, 5, 7, 9, 2, 2])
print("tăng dần:", asc, "| miễn phí:", asc[-3::-3])
print(sum(asc) - sum(asc[-3::-3]))

tăng dần: [2, 2, 5, 6, 7, 9] | miễn phí: [6, 2]
23


## 9. Cách 4 — Đếm (counting sort), tận dụng `cost[i] <= 100`

Vì giá chỉ từ 1 đến 100, ta đếm số viên mỗi giá vào mảng `cnt[0..100]`, rồi duyệt giá **từ 100 xuống 1** — tương đương duyệt dãy đã sắp giảm dần, nhưng không cần sort. Biến `pos` đếm vị trí trong dãy giảm dần ảo.

Đây là cách `O(n + V)` với `V = 100` — minh họa ý tưởng **sắp xếp đếm** khi miền giá trị nhỏ.

In [18]:
def minimum_cost_counting(cost: list[int]) -> int:
    cnt = [0] * 101
    for c in cost:
        cnt[c] += 1
    total = 0
    pos = 0                      # vị trí trong dãy giảm dần
    for price in range(100, 0, -1):
        for _ in range(cnt[price]):
            if pos % 3 != 2:
                total += price
            pos += 1
    return total

print(minimum_cost_counting([6, 5, 7, 9, 2, 2]), minimum_cost_counting([1, 2, 3]), minimum_cost_counting([5, 5]))

23 5 10


`range(100, 0, -1)` đếm **lùi**: 100, 99, ..., 1 (dừng trước 0).

## 10. Vì sao greedy đúng? (exchange argument)

Sắp giảm dần `c1 ≥ c2 ≥ ... ≥ cn`. Xét số viên được tặng tối đa: mỗi viên tặng cần 2 viên mua riêng, nên **nhiều nhất** có `⌊n/3⌋` viên tặng.

**Chặn trên cho từng viên tặng:** giả sử một cách mua bất kỳ tặng các viên `f1 ≥ f2 ≥ ... ≥ fm`. Viên `f_t` (lớn thứ `t` trong các viên tặng) cần: có `t` viên tặng ≥ nó (gồm chính nó) và mỗi viên tặng ≥ nó lại "kèm" 2 viên mua ≥ nó. Vậy có ít nhất `3t` viên có giá ≥ `f_t` → `f_t ≤ c_{3t}`.

**Greedy đạt đúng chặn trên:** greedy tặng `c3, c6, c9, ...` — tức `f_t = c_{3t}` với mọi `t ≤ ⌊n/3⌋`, và hợp lệ vì `c_{3t} ≤ min(c_{3t-2}, c_{3t-1})`.

→ Tổng tiền tặng của greedy ≥ mọi cách khác → tổng tiền trả nhỏ nhất. ∎

Trực giác "trao đổi": nếu một phương án tặng viên rẻ trong khi có thể tặng viên đắt hơn, ta đổi chỗ chúng mà không vi phạm điều kiện và tiết kiệm được nhiều hơn.

## 11. Độ phức tạp

Gọi `n = len(cost)`, `V = 100` (giá lớn nhất).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force hoán vị | `O(n · n!)` | `O(n)` | chỉ để kiểm chứng |
| 2. Sort giảm + `i % 3` | `O(n log n)` | `O(n)` (list sắp mới) | lời giải chính |
| 3. `sum - sum(desc[2::3])` | `O(n log n)` | `O(n)` hoặc `O(1)` thêm nếu sort tại chỗ* | ngắn nhất |
| 4. Counting | `O(n + V)` | `O(V)` | tận dụng miền giá nhỏ |

*Timsort của Python có thể cần tới `O(n)` bộ nhớ tạm khi sort; "tại chỗ" nghĩa là không tạo list kết quả mới.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Ba ví dụ của đề.
2. `n = 1`, `n = 2` (không tặng được gì).
3. Giá trùng nhau (`[3,3,3]` → tặng được viên 3 vì `3 ≤ 3`).
4. `n` chia 3 dư 1, dư 2.
5. Đối chiếu với brute force trên input ngẫu nhiên nhỏ.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS ({actual})")

impls = [minimum_cost_greedy, minimum_cost_slice, minimum_cost_counting,
         lambda c: Solution().minimumCost(list(c))]
cases = [
    ("đề ví dụ 1", [1, 2, 3], 5),
    ("đề ví dụ 2", [6, 5, 7, 9, 2, 2], 23),
    ("đề ví dụ 3", [5, 5], 10),
    ("n = 1", [7], 7),
    ("giá bằng nhau", [3, 3, 3], 6),
    ("dư 1", [4, 3, 2, 1], 8),       # 4+3 tặng 2, trả 1 -> 8
    ("dư 2", [5, 4, 3, 2, 1], 12),   # 5+4 tặng 3, trả 2+1
    ("đã sắp tăng", [1, 2, 3, 4, 5, 6], 16),
]
for name, c, exp in cases:
    for f in impls:
        assert f(list(c)) == exp, (name, f)
    check(name, minimum_cost_greedy(c), exp)

đề ví dụ 1             PASS (5)
đề ví dụ 2             PASS (23)
đề ví dụ 3             PASS (10)
n = 1                  PASS (7)
giá bằng nhau          PASS (6)
dư 1                   PASS (8)
dư 2                   PASS (12)
đã sắp tăng            PASS (16)


In [20]:
import random
rng = random.Random(2144)
checked = 0
for _ in range(300):
    n = rng.randint(1, 7)
    c = [rng.randint(1, 10) for _ in range(n)]
    expected = minimum_cost_bruteforce(c)
    for f in impls:
        assert f(list(c)) == expected, (c, f)
    checked += 1
print("Đối chiếu brute force trên", checked, "input ngẫu nhiên: PASS")

Đối chiếu brute force trên 300 input ngẫu nhiên: PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Sắp **tăng dần** rồi bỏ qua mỗi viên thứ 3 tính từ đầu

In [21]:
def wrong_ascending(cost):
    asc = sorted(cost)
    return sum(x for i, x in enumerate(asc) if i % 3 != 2)

c = [1, 2, 3, 4]
print("sai :", wrong_ascending(c), "| đúng:", minimum_cost_greedy(c))

sai : 7 | đúng: 8


**Vì sao sai:** với `[1,2,3,4]`, cách sai "tặng" viên 3 trong nhóm `(1,2,3)` — vi phạm luật `3 ≤ min(1,2)`! Kết quả `7` nhỏ hơn đáp án đúng `8` nhưng là một cách mua **không hợp lệ**. Đây là loại lỗi nguy hiểm: số "đẹp hơn" nhưng sai luật. **Sửa:** sắp **giảm dần** (hoặc dùng `asc[-3::-3]`).

### Lỗi 2: Tặng viên **rẻ nhất** thay vì viên đắt thứ ba

In [22]:
def wrong_cheapest_free(cost):
    desc = sorted(cost, reverse=True)
    free = len(cost) // 3
    return sum(desc) - sum(sorted(cost)[:free])   # tặng các viên rẻ nhất

c = [6, 5, 7, 9, 2, 2]
print("sai :", wrong_cheapest_free(c), "| đúng:", minimum_cost_greedy(c))

sai : 27 | đúng: 23


**Vì sao sai:** muốn trả ít thì phải được **tặng nhiều tiền nhất** — tặng viên rẻ là tiết kiệm ít. **Sửa:** viên tặng là `desc[2], desc[5], ...`.

### Lỗi 3: Dùng `i % 3 == 0` (lệch vị trí)

In [23]:
def wrong_offset(cost):
    desc = sorted(cost, reverse=True)
    return sum(x for i, x in enumerate(desc) if i % 3 != 0)

c = [1, 2, 3]
print("sai :", wrong_offset(c), "| đúng:", minimum_cost_greedy(c))

sai : 3 | đúng: 5


**Vì sao sai:** `i % 3 == 0` ứng với viên **đầu** nhóm (đắt nhất) — không thể được tặng. Viên tặng là viên **thứ ba**: chỉ số `2, 5, 8` ⇒ `i % 3 == 2`.

### Lỗi 4: Quên rằng `list.sort()` trả `None`

In [24]:
c = [3, 1, 2]
result = c.sort(reverse=True)
print("result =", result, "| c =", c)
try:
    print(sum(result))
except TypeError as e:
    print("TypeError:", e)

result = None | c = [3, 2, 1]
TypeError: 'NoneType' object is not iterable


## 14. Mẹo nhận dạng pattern

- "Mua k tặng 1", "chọn để **tối đa** phần tiết kiệm/tối thiểu chi phí", có điều kiện "phần được tặng ≤ ..." → **sắp xếp + greedy theo nhóm**.
- Cấu trúc "cứ mỗi nhóm `m` phần tử" sau khi sắp → `enumerate` + `i % m`, hoặc slicing `a[m-1::m]`.
- Miền giá trị nhỏ (≤ 100, ≤ 1000) → có thể thay sort bằng **counting**.
- Luôn chuẩn bị **brute force nhỏ** để kiểm chứng greedy — greedy rất dễ "có vẻ đúng" mà sai.

Bài liên quan: #561 Array Partition (sort, lấy mỗi phần tử thứ 2), #1561 Maximum Number of Coins You Can Get (sort, nhóm 3), #2171 Removing Minimum Number of Magic Beans.

## 15. Biến thể

### 15.1. "Mua `k` tặng 1" tổng quát
Sắp giảm dần, viên ở vị trí `k, 2k+1, ...` (tức `i % (k+1) == k`) miễn phí.

In [25]:
def min_cost_buy_k_get_one(cost, k):
    desc = sorted(cost, reverse=True)
    return sum(x for i, x in enumerate(desc) if i % (k + 1) != k)

print(min_cost_buy_k_get_one([6, 5, 7, 9, 2, 2], 2))   # bài gốc: 23
print(min_cost_buy_k_get_one([6, 5, 7, 9, 2, 2], 1))   # mua 1 tặng 1

23
17


### 15.2. Trả về cả danh sách viên được tặng (để in hóa đơn)

In [26]:
def plan_purchase(cost):
    desc = sorted(cost, reverse=True)
    groups = [desc[g:g + 3] for g in range(0, len(desc), 3)]
    plan = []
    for grp in groups:
        if len(grp) == 3:
            plan.append({"mua": grp[:2], "tặng": grp[2]})
        else:
            plan.append({"mua": grp, "tặng": None})
    return plan

for line in plan_purchase([6, 5, 7, 9, 2, 2, 8]):
    print(line)

{'mua': [9, 8], 'tặng': 7}
{'mua': [6, 5], 'tặng': 2}
{'mua': [2], 'tặng': None}


### 15.3. LeetCode 1561 — Maximum Number of Coins You Can Get (cùng tư duy nhóm 3)
Có `3n` đống xu; mỗi lượt chọn 3 đống, Alice lấy đống lớn nhất, bạn lấy đống lớn nhì, Bob lấy đống nhỏ nhất. Tối đa số xu của bạn: sắp tăng dần, Bob nhận `n` đống nhỏ nhất, bạn lấy mỗi đống thứ 2 từ cuối.

In [27]:
def max_coins(piles):
    piles = sorted(piles)
    n = len(piles) // 3
    return sum(piles[n::2])

print(max_coins([2, 4, 1, 2, 7, 8]))   # 9

9


## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Tính tay
`cost = [10, 1, 1, 10, 5, 5, 5]`. Sắp giảm dần, đánh dấu viên miễn phí, tính tổng tiền.

### Bài 2 — Đoán output
`sorted([4, 8, 1, 9, 3], reverse=True)[2::3]` = ?

### Bài 3 — Điền chỗ trống

In [28]:
def exercise_minimum_cost(cost: list[int]) -> int:
    # TODO 1: sắp giảm dần
    # TODO 2: duyệt i, x; nếu i % 3 == 2 thì bỏ qua
    # TODO 3: còn lại cộng vào total
    pass

# assert exercise_minimum_cost([6, 5, 7, 9, 2, 2]) == 23
# assert exercise_minimum_cost([5, 5]) == 10

### Bài 4 — Tổng tiền **tiết kiệm** được
Trả về số tiền khách tiết kiệm được so với không khuyến mãi.

In [29]:
def exercise_saving(cost: list[int]) -> int:
    # TODO
    pass

# assert exercise_saving([6, 5, 7, 9, 2, 2]) == 8

### Bài 5 — Biến thể ngân sách
Với ngân sách `budget`, khách mua theo chiến lược greedy (theo từng nhóm 3 viên đắt → rẻ, phải trả đủ một nhóm mới lấy được nhóm đó). Hỏi **lấy được tối đa bao nhiêu nhóm đầy đủ** trước khi hết tiền? (Chỉ tính các nhóm theo thứ tự giảm dần.)

In [30]:
def exercise_groups_within_budget(cost: list[int], budget: int) -> int:
    # TODO: chia desc thành nhóm 3, cộng dồn tiền (2 viên đầu mỗi nhóm), đếm nhóm còn đủ tiền
    pass

# assert exercise_groups_within_budget([6, 5, 7, 9, 2, 2], 16) == 1
# assert exercise_groups_within_budget([6, 5, 7, 9, 2, 2], 15) == 0

### Bài 6 — Mini challenge: counting không cần biến `pos` riêng
Viết lại Cách 4 sao cho chỉ dùng mảng đếm và **một** vòng duyệt giá; với mỗi giá, tính trực tiếp có bao nhiêu viên giá đó rơi vào vị trí `% 3 == 2`.

In [31]:
def exercise_counting_fast(cost: list[int]) -> int:
    # TODO: với mỗi price (giảm dần) có cnt viên chiếm các vị trí pos..pos+cnt-1
    #       đếm số vị trí p trong đoạn đó có p % 3 == 2 bằng công thức
    pass

# assert exercise_counting_fast([6, 5, 7, 9, 2, 2]) == 23

### Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Sắp giảm dần trước | `[10,10,5,5,5,1,1]` | viên ở chỉ số 2 và 5 miễn phí |
| 2 | `[::3]` bắt đầu từ chỉ số 2 | dãy giảm dần là `[9,8,4,3,1]` | chỉ số 2 → `4` (chỉ số 5 không tồn tại) |
| 3 | Viên thứ 3 mỗi nhóm free | `enumerate(sorted(..., reverse=True))` | `if i % 3 == 2: continue` |
| 4 | Tiết kiệm = tổng các viên free | `desc[2::3]` | `sum(desc[2::3])` |
| 5 | Nhóm `g` cần trả `desc[3g] + desc[3g+1]` | cộng dồn rồi so với budget | nhóm thiếu (len < 3) không tính |
| 6 | Số `p` trong `[0, x)` có `p%3==2` là `x // 3` | đếm trong `[pos, pos+cnt)` = f(pos+cnt) - f(pos) | `free = f(pos+cnt) - f(pos)`, `total += price*(cnt-free)` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** giảm dần `[10,10,5,5,5,1,1]`; miễn phí chỉ số 2 (`5`) và 5 (`1`); trả `10+10+5+5+1 = 31`.

**Bài 2:** `[9,8,4,3,1][2::3] = [4]`.

In [32]:
trace_minimum_cost([10, 1, 1, 10, 5, 5, 5])
print(sorted([4, 8, 1, 9, 3], reverse=True)[2::3])

cost = [10, 1, 1, 10, 5, 5, 5] -> giảm dần [10, 10, 5, 5, 5, 1, 1]
i=0: x=10 trả tiền                       | total=10
i=1: x=10 trả tiền                       | total=20
i=2: x=5 miễn phí vì 5 <= min(10, 10) = 10 | total=20
i=3: x=5 trả tiền                       | total=25
i=4: x=5 trả tiền                       | total=30
i=5: x=1 miễn phí vì 1 <= min(5, 5) = 5 | total=30
i=6: x=1 trả tiền                       | total=31
Kết quả: 31
[4]


In [33]:
def answer_minimum_cost(cost):
    total = 0
    for i, x in enumerate(sorted(cost, reverse=True)):
        if i % 3 == 2:
            continue
        total += x
    return total


def answer_saving(cost):
    return sum(sorted(cost, reverse=True)[2::3])


def answer_groups_within_budget(cost, budget):
    desc = sorted(cost, reverse=True)
    spent = 0
    groups = 0
    for g in range(0, len(desc) - 2, 3):          # chỉ nhóm đủ 3 viên
        spent += desc[g] + desc[g + 1]
        if spent > budget:
            break
        groups += 1
    return groups


def answer_counting_fast(cost):
    cnt = [0] * 101
    for c in cost:
        cnt[c] += 1
    f = lambda x: x // 3        # số p trong [0, x) có p % 3 == 2
    total = 0
    pos = 0
    for price in range(100, 0, -1):
        if cnt[price]:
            free = f(pos + cnt[price]) - f(pos)
            total += price * (cnt[price] - free)
            pos += cnt[price]
    return total


assert answer_minimum_cost([6, 5, 7, 9, 2, 2]) == 23
assert answer_minimum_cost([5, 5]) == 10
assert answer_saving([6, 5, 7, 9, 2, 2]) == 8
assert answer_groups_within_budget([6, 5, 7, 9, 2, 2], 16) == 1
assert answer_groups_within_budget([6, 5, 7, 9, 2, 2], 15) == 0
assert answer_groups_within_budget([6, 5, 7, 9, 2, 2], 23) == 2
rng = random.Random(1)
for _ in range(300):
    c = [rng.randint(1, 100) for _ in range(rng.randint(1, 40))]
    assert answer_counting_fast(c) == minimum_cost_greedy(c) == answer_minimum_cost(c)
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minimumCost(self, cost: list[int]) -> int:
        cost = sorted(cost, reverse=True)
        return sum(cost) - sum(cost[2::3])
```

**Khung tư duy greedy theo nhóm:** sắp xếp theo thứ tự "có lợi" → gom nhóm cố định → xác định vị trí được lợi trong nhóm (`i % m`) → chứng minh bằng chặn trên/trao đổi → đối chiếu brute force.

**Checklist trước khi nộp:**
- [ ] Sắp **giảm dần** (không phải tăng dần)?
- [ ] Viên miễn phí ở chỉ số `2, 5, 8, ...` (`i % 3 == 2`)?
- [ ] `n < 3` không có viên miễn phí?
- [ ] Không dùng giá trị trả về của `list.sort()`?
- [ ] Đã thử với giá trùng nhau?